### Average planned (GTFS) vs actual (SIRI) schedule adherence, across many days

This is a multi-day extension of `compare gtfs planned vs siri actual.ipynb`. Instead of comparing one ride's plan vs actual, it:

1. Scans backward day by day from a reference date, looking for a ride of `LINE_REF` scheduled at the exact same time of day (`TIME_OF_DAY`) on each day.
2. Keeps only days whose planned stop sequence exactly matches a "canonical" reference sequence - i.e. same route, same stops, same direction - discarding any day where the route shape differs.
3. Requires at least `MIN_RIDES` such matched days (with both planned and actual data) before proceeding.
4. For each matched day, snaps its actual SIRI pings to the canonical stops and computes elapsed minutes since that day's own scheduled start (so day-to-day absolute time differences don't matter - everything is on a shared "minutes since departure" axis).
5. Averages the actual elapsed time per stop across all matched days (ignoring stops with no data on a given day), and plots it against the canonical planned schedule (also shifted so departure = minute 0).

Requires `pip install open-bus-stride-client`.


In [1]:
!pip install open-bus-stride-client folium branca tqdm


[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dateutil import tz
from tqdm import tqdm

pd.options.display.max_columns = 1000
pd.options.display.max_colwidth = 1000

import stride

### Configuration

Edit these once - every cell below reuses them.

- `TIME_OF_DAY`: the wall-clock departure time to match across days - use whatever exact minute this `LINE_REF` actually departs at, found via the lookup cell below.
- `MIN_RIDES`: minimum matched days required before proceeding (default 20). There's no fixed target count to gather - every day within `MAX_DAYS_TO_SCAN` that has both a matching plan and actual ride is kept, so the true count depends entirely on `LINE_REF`'s real history (printed by the scan cell below).
- `MAX_DAYS_TO_SCAN`: how far back to look for candidate days - default 90 (~3 months), the full range this environment's data actually covers.
- `MATCH_TOLERANCE_MIN`: an actual GPS ping only snaps to a planned stop if their elapsed-since-start times are within this many minutes of each other. Without this, a route that loops back near its own path (very common) can snap a much-later ping to an early stop just because it happens to be geometrically close - producing a nonsensical "70 minutes late" outlier that isn't real. Restricting candidates to a plausible time window first, then picking the nearest by distance, avoids that.

In [3]:
# ---- pool of known well-covered bus lines ----
# found by scanning /rides_execution/list for many bus lines over the last 90 days and keeping
# ones with a full 90-day history AND a high (>95%) ride-level tracked rate
GOOD_LINE_REFS = ['18663', '40499', '29099', '10208', '11603', '29620', '40089', '36716']

import random
i = random.randrange(len(GOOD_LINE_REFS))
LINE_REF = GOOD_LINE_REFS[i]
print(f"sampled LINE_REF = {LINE_REF} (index {i} of {len(GOOD_LINE_REFS)})")

# ---- date range ----
REFERENCE_DATE = datetime.date(2026, 7, 30)   # start scanning backward from here
MAX_DAYS_TO_SCAN = 90                          # ~3 months - the full range this environment's data actually covers

TZ = tz.gettz('Israel')

# ---- derive operator/agency/route metadata for whichever LINE_REF got sampled above ----
_route_info = stride.get('/gtfs_routes/list', {
    'line_refs': LINE_REF, 'date_from': REFERENCE_DATE.isoformat(), 'date_to': REFERENCE_DATE.isoformat(),
})
if not _route_info:
    # today sometimes has no data yet in this environment - fall back to yesterday
    _route_info = stride.get('/gtfs_routes/list', {
        'line_refs': LINE_REF,
        'date_from': (REFERENCE_DATE - datetime.timedelta(days=1)).isoformat(),
        'date_to': (REFERENCE_DATE - datetime.timedelta(days=1)).isoformat(),
    })
_route_info = _route_info[0]
OPERATOR_REF = _route_info['operator_ref']
AGENCY_NAME = _route_info['agency_name']
ROUTE_SHORT_NAME = _route_info['route_short_name']

# ---- derive a recurring departure time for whichever LINE_REF got sampled above ----
# use the earliest departure on the most recent day that actually has data for this line
for _days_back in range(1, 10):
    _day = REFERENCE_DATE - datetime.timedelta(days=_days_back)
    _rows = stride.get('/route_timetable/list', {
        'line_refs': LINE_REF,
        'planned_start_time_date_from': datetime.datetime.combine(_day, datetime.time(0, 0), tzinfo=TZ),
        'planned_start_time_date_to': datetime.datetime.combine(_day, datetime.time(23, 59), tzinfo=TZ),
        'order_by': 'gtfs_line_start_time',
        'limit': 1,
    })
    if _rows:
        TIME_OF_DAY = pd.to_datetime(_rows[0]['gtfs_line_start_time']).tz_convert('Israel').time()
        break
else:
    raise RuntimeError(f"no data found for LINE_REF {LINE_REF} in the last 10 days - pick a different line")

print(f"LINE_REF {LINE_REF}: operator_ref={OPERATOR_REF}, agency={AGENCY_NAME}, "
      f"route_short_name={ROUTE_SHORT_NAME}, TIME_OF_DAY={TIME_OF_DAY}")

# ---- how many matched days to gather ----
# No fixed target count - every day in the scanned window that has both a matching plan and
# actual data is kept, so the true number of matched days depends on LINE_REF's real history.
MIN_RIDES = 20            # minimum required to proceed with the average

# ---- matching / request tuning ----
MATCH_TOLERANCE_MIN = 20  # see note above
REQUEST_LIMIT = 15000     # explicit server-side limit - see the companion notebook for why this matters

def localize_dates(data, dt_columns=None):
    data = data.copy()
    for c in (dt_columns or []):
        data[c] = pd.to_datetime(data[c]).dt.tz_convert('Israel')
    return data

sampled LINE_REF = 40089 (index 6 of 8)
LINE_REF 40089: operator_ref=18, agency=קווים, route_short_name=168, TIME_OF_DAY=00:05:00


### (Optional) Find LINE_REF / OPERATOR_REF / TIME_OF_DAY for a different line

Look up a line's route variants and their scheduled departure times, then copy the ones you want into the configuration cell above.

In [4]:
pd.DataFrame(stride.get('/gtfs_routes/list', {
    'route_short_name': ROUTE_SHORT_NAME,
    'operator_refs': OPERATOR_REF,
    'agency_name': AGENCY_NAME,
    'date_from': REFERENCE_DATE.isoformat(),
    'date_to': (REFERENCE_DATE + datetime.timedelta(days=1)).isoformat(),
    'limit': REQUEST_LIMIT,
}))

,id,date,line_ref,operator_ref,route_short_name,route_long_name,route_mkt,route_direction,route_alternative,agency_name,route_type
0,9345346,2026-07-30,40088,18,168,קדושת לוי/שלום רב-ביתר עילית<->שדרות האמוראים/רבינא-בית שמש-10,15168,1,0,קווים,3
1,9345347,2026-07-30,40089,18,168,שדרות האמוראים/זכריה הנביא-בית שמש<->הר''ן/קדושת לוי-ביתר עילית-20,15168,2,0,קווים,3


### Scan backward day by day, matching plan + actual for the same time of day

`route_timetable/list` rejects any request spanning more than 1 day, so each candidate day is queried individually with a tight 1-minute window around `TIME_OF_DAY`. A day only counts if:
- a planned ride exists at exactly that time,
- its stop sequence matches the canonical one (the first matched day's sequence) - same route, same stops, same direction,
- and actual SIRI data exists for that same ride.

If a day's plan has more than one distinct `gtfs_ride_id` (or actual has more than one distinct `siri_ride__id`) in that 1-minute window, only the larger group is kept, to avoid silently mixing two different rides.

In [5]:
def largest_group(df, id_column):
    if df[id_column].nunique() <= 1:
        return df
    largest_id = df[id_column].value_counts().idxmax()
    return df[df[id_column] == largest_id]

canonical_stops = None   # tuple of stop names - defines "same route, same stops, same direction"
canonical_plan = None    # the reference day's plan dataframe
valid_days = []          # list of {'date', 'start', 'actual'} for every matched day
skipped = {'no_plan': 0, 'route_mismatch': 0, 'no_actual': 0}

progress = tqdm(range(MAX_DAYS_TO_SCAN), desc="scanning days")
for days_back in progress:
    day = REFERENCE_DATE - datetime.timedelta(days=days_back)
    window_from = datetime.datetime.combine(day, TIME_OF_DAY, tzinfo=TZ)
    window_to = window_from + datetime.timedelta(minutes=1)

    plan_rows = stride.get('/route_timetable/list', {
        'line_refs': LINE_REF,
        'planned_start_time_date_from': window_from,
        'planned_start_time_date_to': window_to,
        'order_by': 'planned_arrival_time',
        'limit': REQUEST_LIMIT,
    })
    if not plan_rows:
        skipped['no_plan'] += 1
        progress.set_postfix(matched=len(valid_days), **skipped)
        continue

    plan = localize_dates(pd.DataFrame(plan_rows), ['planned_arrival_time', 'gtfs_line_start_time'])
    plan = largest_group(plan, 'gtfs_ride_id').sort_values('planned_arrival_time').reset_index(drop=True)
    stop_signature = tuple(plan['name'])

    if canonical_stops is None:
        canonical_stops = stop_signature
        canonical_plan = plan
    elif stop_signature != canonical_stops:
        skipped['route_mismatch'] += 1
        progress.set_postfix(matched=len(valid_days), **skipped)
        continue

    actual_rows = stride.get('/siri_vehicle_locations/list', {
        'siri_routes__line_ref': LINE_REF,
        'siri_routes__operator_ref': OPERATOR_REF,
        'siri_rides__schedualed_start_time_from': window_from,
        'siri_rides__schedualed_start_time_to': window_to,
        'order_by': 'recorded_at_time',
        'limit': REQUEST_LIMIT,
    })
    if not actual_rows:
        skipped['no_actual'] += 1
        progress.set_postfix(matched=len(valid_days), **skipped)
        continue

    actual = localize_dates(pd.DataFrame(actual_rows), ['recorded_at_time', 'siri_ride__scheduled_start_time'])
    actual = largest_group(actual, 'siri_ride__id').sort_values('recorded_at_time').reset_index(drop=True)

    valid_days.append({'date': day, 'start': window_from, 'actual': actual})
    progress.set_postfix(matched=len(valid_days), **skipped)

assert len(valid_days) >= MIN_RIDES, (
    f"only found {len(valid_days)} matched days (need >= {MIN_RIDES}) after scanning {MAX_DAYS_TO_SCAN} days back "
    f"from {REFERENCE_DATE} - skipped: {skipped}. Lower MIN_RIDES, raise MAX_DAYS_TO_SCAN, or pick a "
    f"LINE_REF/TIME_OF_DAY with more historical data in this environment."
)

print(f"Scanned {MAX_DAYS_TO_SCAN} days back from {REFERENCE_DATE} - "
      f"found {len(valid_days)} matched days out of {MAX_DAYS_TO_SCAN} scanned "
      f"(no fixed target - this is every day in the window with both a matching plan and actual data)")
print(f"matched days: {[d['date'].isoformat() for d in valid_days]}")
print(f"canonical route: {len(canonical_plan)} stops")
print(f"skipped: {skipped}")

scanning days: 100%|██████████| 90/90 [02:10<00:00,  1.45s/it, matched=3, no_actual=1, no_plan=86, route_mismatch=0]


AssertionError: only found 3 matched days (need >= 20) after scanning 90 days back from 2026-07-30 - skipped: {'no_plan': 86, 'route_mismatch': 0, 'no_actual': 1}. Lower MIN_RIDES, raise MAX_DAYS_TO_SCAN, or pick a LINE_REF/TIME_OF_DAY with more historical data in this environment.

### Snap each day's actual pings to the canonical stops, and average

Each matched day's SIRI pings are snapped to whichever canonical stop is nearest by lon/lat *and* plausible in time (within `MATCH_TOLERANCE_MIN` minutes of that stop's planned elapsed time) - see the config note above for why the time constraint matters. Elapsed time is measured from each day's own scheduled start, so day 0 is always "departure" regardless of which day it actually was. The average across days ignores stops with no matched ping on a given day (`skipna=True`), rather than treating missing data as zero.

In [ ]:
canonical_plan_elapsed = (
    (canonical_plan['planned_arrival_time'] - canonical_plan['gtfs_line_start_time']).dt.total_seconds() / 60
)
canonical_xy = canonical_plan[['lon', 'lat']].to_numpy()
canonical_elapsed_arr = canonical_plan_elapsed.to_numpy()

per_day_stop_elapsed = []
for day in valid_days:
    actual = day['actual']
    actual_xy = actual[['lon', 'lat']].to_numpy()
    elapsed_actual = ((actual['recorded_at_time'] - day['start']).dt.total_seconds() / 60).to_numpy()

    dists = np.linalg.norm(actual_xy[:, None, :] - canonical_xy[None, :, :], axis=2)
    time_gap = np.abs(elapsed_actual[:, None] - canonical_elapsed_arr[None, :])
    dists_masked = np.where(time_gap <= MATCH_TOLERANCE_MIN, dists, np.inf)

    nearest_idx = dists_masked.argmin(axis=1)
    has_match = np.isfinite(dists_masked.min(axis=1))

    per_stop = pd.Series(elapsed_actual[has_match], index=nearest_idx[has_match]).groupby(level=0).mean()
    per_day_stop_elapsed.append(per_stop.reindex(range(len(canonical_plan))))

elapsed_by_day = pd.concat(per_day_stop_elapsed, axis=1)
elapsed_by_day.columns = [d['date'].isoformat() for d in valid_days]

average_actual_elapsed = elapsed_by_day.mean(axis=1, skipna=True)

elapsed_by_day.shape, average_actual_elapsed.notna().sum()

### Plot: canonical plan vs. average actual, by stop

x-axis is stop index along the canonical route, y-axis is minutes elapsed since departure (0 at the first stop). The faint lines are each individual matched day (context on spread), the bold lines are the canonical plan and the cross-day average actual.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

stop_idx = range(len(canonical_plan))
for day_col in elapsed_by_day.columns:
    ax.plot(stop_idx, elapsed_by_day[day_col], color='tab:orange', alpha=0.15, linewidth=1, zorder=1)

ax.plot(stop_idx, canonical_plan_elapsed, linestyle='dashed', marker='o', markerfacecolor='none',
        color='tab:purple', linewidth=2, label='Planned (canonical GTFS)', zorder=3)
ax.plot(stop_idx, average_actual_elapsed, linestyle='solid', marker='o',
        color='tab:orange', markeredgecolor='black', linewidth=2,
        label=f'Average actual (SIRI, n={len(valid_days)} days)', zorder=2)

ax.set_xlabel('Stop index (sequence order along the route)')
ax.set_ylabel('Minutes since departure')
ax.set_title(f"Schedule adherence average - line_ref {LINE_REF}, {TIME_OF_DAY.strftime('%H:%M')} "
             f"departure, {len(valid_days)} days", fontsize=10)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()

plt.show()

### Plot planned vs. averaged-actual on an actual map

Same idea as the stringline chart, but on a real map, and with one more layer of averaging: the planned route uses each stop's fixed GTFS coordinate, but the "averaged actual" route does **not** reuse those coordinates - instead, every matched SIRI ping across every day is pooled by which stop it snapped to, and both its **position** (lon/lat) and its **elapsed time** are averaged together, weighted by `1 / distance_to_stop` (a ping recorded right at the stop counts more than one that just barely qualified within `MATCH_TOLERANCE_MIN`). So the averaged-actual route is a genuinely independent estimate of where and when the bus tends to actually be - not just the plan re-colored. Marker size on the actual route reflects how many pings contributed to that stop's average (more pings = bigger, more confident average).

In [ ]:
import folium
import branca.colormap as cm

n_stops = len(canonical_plan)
pooled = {i: {'lon': [], 'lat': [], 'elapsed': [], 'weight': []} for i in range(n_stops)}

for day in valid_days:
    actual = day['actual']
    actual_xy = actual[['lon', 'lat']].to_numpy()
    elapsed_actual = ((actual['recorded_at_time'] - day['start']).dt.total_seconds() / 60).to_numpy()

    dists = np.linalg.norm(actual_xy[:, None, :] - canonical_xy[None, :, :], axis=2)
    time_gap = np.abs(elapsed_actual[:, None] - canonical_elapsed_arr[None, :])
    dists_masked = np.where(time_gap <= MATCH_TOLERANCE_MIN, dists, np.inf)

    nearest_idx = dists_masked.argmin(axis=1)
    nearest_dist = dists_masked.min(axis=1)
    has_match = np.isfinite(nearest_dist)

    for j in np.where(has_match)[0]:
        s = int(nearest_idx[j])
        weight = 1.0 / (nearest_dist[j] + 1e-6)  # pings closer to the stop count more
        pooled[s]['lon'].append(actual_xy[j, 0])
        pooled[s]['lat'].append(actual_xy[j, 1])
        pooled[s]['elapsed'].append(elapsed_actual[j])
        pooled[s]['weight'].append(weight)

def weighted_mean(values, weights):
    return float(np.average(values, weights=weights)) if values else np.nan

avg_actual_lon = np.array([weighted_mean(pooled[i]['lon'], pooled[i]['weight']) for i in range(n_stops)])
avg_actual_lat = np.array([weighted_mean(pooled[i]['lat'], pooled[i]['weight']) for i in range(n_stops)])
avg_actual_elapsed = np.array([weighted_mean(pooled[i]['elapsed'], pooled[i]['weight']) for i in range(n_stops)])
n_pings_per_stop = np.array([len(pooled[i]['lon']) for i in range(n_stops)])

has_avg = ~np.isnan(avg_actual_lon)
print(f"{has_avg.sum()}/{n_stops} stops have a weighted spatial+temporal average "
      f"(pooled from {int(n_pings_per_stop.sum())} matched pings across {len(valid_days)} days)")

vmax = max(canonical_plan_elapsed.max(), np.nanmax(avg_actual_elapsed))
colormap = cm.LinearColormap(colors=['#440154', '#31688e', '#35b779', '#fde725'], vmin=0, vmax=vmax)
colormap.caption = 'Minutes since departure'

route_map = folium.Map(location=[canonical_plan['lat'].mean(), canonical_plan['lon'].mean()],
                        zoom_start=13, tiles='OpenStreetMap', width=900, height=600)

# planned route: dashed, hollow markers at each stop's fixed GTFS coordinate
plan_coords = list(zip(canonical_plan['lat'], canonical_plan['lon']))
for (lat1, lon1), (lat2, lon2), t in zip(plan_coords[:-1], plan_coords[1:], canonical_plan_elapsed.iloc[:-1]):
    folium.PolyLine([(lat1, lon1), (lat2, lon2)], color=colormap(t), weight=4, opacity=0.9,
                     dash_array='6,6').add_to(route_map)
for (lat, lon), t, name in zip(plan_coords, canonical_plan_elapsed, canonical_plan['name']):
    folium.CircleMarker(location=(lat, lon), radius=6, color=colormap(t), weight=2, fill=False,
                         popup=f"Planned: {name} @ {t:.1f} min").add_to(route_map)

# averaged actual route: solid, filled markers at the weighted-average GPS position (not the
# canonical stop location) and weighted-average elapsed time - sized by how many pings contributed
avg_coords = list(zip(avg_actual_lat[has_avg], avg_actual_lon[has_avg]))
avg_elapsed_valid = avg_actual_elapsed[has_avg]
avg_n_valid = n_pings_per_stop[has_avg]
avg_stop_idx = np.where(has_avg)[0]

for (lat1, lon1), (lat2, lon2), t in zip(avg_coords[:-1], avg_coords[1:], avg_elapsed_valid[:-1]):
    folium.PolyLine([(lat1, lon1), (lat2, lon2)], color=colormap(t), weight=4, opacity=0.9).add_to(route_map)
for (lat, lon), t, n, idx in zip(avg_coords, avg_elapsed_valid, avg_n_valid, avg_stop_idx):
    folium.CircleMarker(
        location=(lat, lon), radius=5 + min(n, 50) ** 0.5, color='black', weight=1, fill=True,
        fill_color=colormap(t), fill_opacity=1.0,
        popup=f"Averaged actual: stop {idx} @ {t:.1f} min (n={n} pings)",
    ).add_to(route_map)

colormap.add_to(route_map)
route_map